# Phase 2 — Nettoyage & Harmonisation
**Data Challenge Agriculture — Togo AI Lab (Défi 2)**

Ce notebook prend les CSV bruts de `data/raw/` et produit des fichiers propres dans `data/processed/`.

Étapes :
1. Rechargement et conversion WKT → GeoDataFrame
2. Détection et suppression des doublons
3. Vérification des bornes géographiques
4. Harmonisation des noms administratifs
5. Extraction des centroïdes
6. Nettoyage des séries Banque mondiale
7. Export vers `data/processed/`

In [1]:
import pandas as pd
import geopandas as gpd
import numpy as np
import re
import unicodedata
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

ROOT = Path('..')
RAW  = ROOT / 'data' / 'raw'
PROC = ROOT / 'data' / 'processed'
PROC.mkdir(parents=True, exist_ok=True)

print('Dossiers OK')
print('RAW :', RAW.resolve())
print('PROC:', PROC.resolve())

Dossiers OK
RAW : C:\Users\admin\Music\Data - Challenge\Agriculture-Challenge\data\raw
PROC: C:\Users\admin\Music\Data - Challenge\Agriculture-Challenge\data\processed


## 0. Fonctions utilitaires

In [2]:
def read_csv_smart(path):
    for enc in ['utf-8', 'utf-8-sig', 'latin-1', 'cp1252']:
        for sep in [',', ';', '\t']:
            try:
                df = pd.read_csv(path, sep=sep, encoding=enc)
                if len(df.columns) > 1:
                    return df, enc, sep
            except Exception:
                continue
    raise ValueError(f'Impossible de lire : {path}')


def wkt_to_geodataframe(df, geom_col='geometry', crs='EPSG:4326'):
    """Convertit colonne WKT en GeoDataFrame + calcule centroïde."""
    gdf = gpd.GeoDataFrame(
        df,
        geometry=gpd.GeoSeries.from_wkt(df[geom_col]),
        crs=crs
    )
    gdf['centroid_lon'] = gdf.geometry.centroid.x
    gdf['centroid_lat'] = gdf.geometry.centroid.y
    return gdf


def normalize_name(s):
    """
    Normalise un nom géographique pour comparaisons :
    - minuscules
    - supprime accents
    - supprime ponctuation et espaces multiples
    - supprime préfixes courants (préfecture de, région de...)
    """
    if pd.isna(s):
        return ''
    s = str(s).lower().strip()
    # Supprime accents
    s = unicodedata.normalize('NFD', s)
    s = ''.join(c for c in s if unicodedata.category(c) != 'Mn')
    # Supprime préfixes courants
    for prefix in ['prefecture de ', 'region de ', 'commune de ', 'canton de ']:
        s = s.replace(prefix, '')
    # Normalise espaces et tirets
    s = re.sub(r'[\-_]+', ' ', s)
    s = re.sub(r'\s+', ' ', s).strip()
    return s


print('Fonctions utilitaires chargées ✓')

Fonctions utilitaires chargées ✓


## 1. Sources géospatiales — nettoyage

Pour chaque source : rechargement → doublons → bornes → colonnes utiles → centroïde.

In [3]:
GEO_FILES = {
    'elevage':      RAW / 'elevage.csv',
    'abattoirs':    RAW / 'abattoirs.csv',
    'pisciculture': RAW / 'pisciculture.csv',
    'retenues_eau': RAW / 'retenues_eau.csv',
    'barrages':     RAW / 'barrages.csv',
}

geo_clean = {}  # GeoDataFrames nettoyés

for name, path in GEO_FILES.items():
    print(f'\n{"="*55}')
    print(f'[{name.upper()}]  {path.name}')

    df, enc, sep = read_csv_smart(path)
    n0 = len(df)

    # ── 1. Suppression doublons sur FID ──────────────────────────
    n_dup_fid = df.duplicated(subset=['FID']).sum()
    if n_dup_fid > 0:
        df = df.drop_duplicates(subset=['FID'])
        print(f'  Doublons FID supprimés : {n_dup_fid}')
    else:
        print(f'  Doublons FID : 0 ✓')

    # ── 2. Doublons géographiques (même géométrie exacte) ────────
    n_dup_geo = df.duplicated(subset=['geometry']).sum()
    if n_dup_geo > 0:
        df = df.drop_duplicates(subset=['geometry'])
        print(f'  Doublons géométrie supprimés : {n_dup_geo}')
    else:
        print(f'  Doublons géométrie : 0 ✓')

    print(f'  Lignes : {n0} → {len(df)} (supprimé {n0 - len(df)})')

    # ── 3. Conversion WKT → GeoDataFrame ─────────────────────────
    gdf = wkt_to_geodataframe(df)
    print(f'  Géométries : {gdf.geom_type.value_counts().to_dict()}')

    # ── 4. Vérification bornes Togo ───────────────────────────────
    hors = gdf[
        (gdf['centroid_lon'] < -0.2) | (gdf['centroid_lon'] > 2.0) |
        (gdf['centroid_lat'] < 5.5)  | (gdf['centroid_lat'] > 11.5)
    ]
    if len(hors) > 0:
        print(f'  ⚠ {len(hors)} géométrie(s) hors bornes Togo — supprimée(s)')
        gdf = gdf[~gdf.index.isin(hors.index)]
    else:
        print(f'  ✓ Toutes les géométries dans les bornes du Togo')

    # ── 5. Normalisation des noms administratifs ──────────────────
    for col in ['region_nom_bdd', 'prefecture_nom_bdd', 'canton_nom_bdd']:
        if col in gdf.columns:
            gdf[col + '_norm'] = gdf[col].apply(normalize_name)

    # ── 6. Ajout colonne source ───────────────────────────────────
    gdf['source'] = name

    geo_clean[name] = gdf
    print(f'  Colonnes finales : {list(gdf.columns)}')
    display(gdf.drop(columns='geometry').head(3))


[ELEVAGE]  elevage.csv
  Doublons FID : 0 ✓
  Doublons géométrie : 0 ✓
  Lignes : 3004 → 3004 (supprimé 0)
  Géométries : {'Polygon': 3004}
  ✓ Toutes les géométries dans les bornes du Togo
  Colonnes finales : ['FID', 'region_nom_bdd', 'prefecture_nom_bdd', 'commune_nom_bdd', 'canton_nom_bdd', 'nom_localite', 'etab_nom', 'etab_adresse', 'etab_jour', 'etab_creation_date', 'activite_statut', 'activite_categorie', 'toilette_type', 'geometry', 'centroid_lon', 'centroid_lat', 'region_nom_bdd_norm', 'prefecture_nom_bdd_norm', 'canton_nom_bdd_norm', 'source']


,FID,region_nom_bdd,prefecture_nom_bdd,commune_nom_bdd,canton_nom_bdd,nom_localite,etab_nom,etab_adresse,etab_jour,etab_creation_date,activite_statut,activite_categorie,toilette_type,centroid_lon,centroid_lat,region_nom_bdd_norm,prefecture_nom_bdd_norm,canton_nom_bdd_norm,source
0,_mview_general_etablissements_elevage.fid-54b5...,Centrale,Blitta,Blitta 1,Blitta,Ditcheyidadi,Ditcheyidadi,Tondja -ditcheyidadi,"{Lundi,Mardi,jeudi,Mercredi,Vendredi,Samedi,Di...",1988,{Construit et Utilise},{Elevage traditionnel},{WCs},0.957211,8.280226,centrale,blitta,blitta,elevage
1,_mview_general_etablissements_elevage.fid-54b5...,Centrale,Blitta,Blitta 1,Blitta,Tondja,Ferme Panouzi,"Blitta gare,tondja","{Lundi,Mardi,Mercredi,jeudi,Vendredi,Dimanche,...",2012,{Construit et Utilise},{Elevage traditionnel},"{Douches,WCs,Latrines seches}",0.958343,8.279474,centrale,blitta,blitta,elevage
2,_mview_general_etablissements_elevage.fid-54b5...,Centrale,Blitta,Blitta 1,Blitta-Village,Tomegbe,Sowougan,Néant,"{Lundi,Mardi,Mercredi,jeudi,Vendredi,Samedi}",2017,{Construit et Utilise},"{Elevage traditionnel,autre}",{autre},1.010151,8.368283,centrale,blitta,blitta village,elevage



[ABATTOIRS]  abattoirs.csv
  Doublons FID : 0 ✓
  Doublons géométrie : 0 ✓
  Lignes : 36 → 36 (supprimé 0)
  Géométries : {'Point': 36}
  ✓ Toutes les géométries dans les bornes du Togo
  Colonnes finales : ['FID', 'region_nom_bdd', 'prefecture_nom_bdd', 'commune_nom_bdd', 'canton_nom_bdd', 'abattoir_nom', 'abattoir_type', 'organisme', 'batiment_fonction', 'geometry', 'centroid_lon', 'centroid_lat', 'region_nom_bdd_norm', 'prefecture_nom_bdd_norm', 'canton_nom_bdd_norm', 'source']


,FID,region_nom_bdd,prefecture_nom_bdd,commune_nom_bdd,canton_nom_bdd,abattoir_nom,abattoir_type,organisme,batiment_fonction,centroid_lon,centroid_lat,region_nom_bdd_norm,prefecture_nom_bdd_norm,canton_nom_bdd_norm,source
0,_mview_agrianimaux_etablissements_abattoir.fid...,Maritime,Agoè-Nyivé,Agoè-Nyivé 4,Togblekope,Aire d'abattage des petits ruminants d'Agoe-Zongo,Traditionnel,Onaf,"Une parc,un bureau, une garde ,les toilettes",1.207853,6.254587,maritime,agoe nyive,togblekope,abattoirs
1,_mview_agrianimaux_etablissements_abattoir.fid...,Maritime,Avé,Avé 1,Tovégan,Togo volaille,autre,Nsp,Nsp,0.920412,6.521438,maritime,ave,tovegan,abattoirs
2,_mview_agrianimaux_etablissements_abattoir.fid...,Maritime,Vo,Vo 1,Vogan,Abattoir de Vogan,Traditionnel,Nsp,Nsp,1.523597,6.340239,maritime,vo,vogan,abattoirs



[PISCICULTURE]  pisciculture.csv
  Doublons FID : 0 ✓
  Doublons géométrie : 0 ✓
  Lignes : 141 → 141 (supprimé 0)
  Géométries : {'MultiPolygon': 141}
  ✓ Toutes les géométries dans les bornes du Togo
  Colonnes finales : ['FID', 'region_nom_bdd', 'prefecture_nom_bdd', 'commune_nom_bdd', 'canton_nom_bdd', 'etab_adr', 'ouverture_jour', 'organisme', 'geometry', 'centroid_lon', 'centroid_lat', 'region_nom_bdd_norm', 'prefecture_nom_bdd_norm', 'canton_nom_bdd_norm', 'source']


,FID,region_nom_bdd,prefecture_nom_bdd,commune_nom_bdd,canton_nom_bdd,etab_adr,ouverture_jour,organisme,centroid_lon,centroid_lat,region_nom_bdd_norm,prefecture_nom_bdd_norm,canton_nom_bdd_norm,source
0,_mview_general_zones_pisiculture.fid-54b5a690_...,Maritime,Avé,Avé 1,Kévé,Keve vogancope,"{Lundi,Mercredi,Mardi,Jeudi,Vendredi,Samedi,Di...",FAPV,1.032488,6.465044,maritime,ave,keve,pisciculture
1,_mview_general_zones_pisiculture.fid-54b5a690_...,Maritime,Avé,Avé 1,Kévé,Keve,"{Lundi,Mardi,Mercredi,Jeudi,Vendredi,Samedi}",LORENOVICH INTERNATIONAL,1.018994,6.463083,maritime,ave,keve,pisciculture
2,_mview_general_zones_pisiculture.fid-54b5a690_...,Maritime,Avé,Avé 1,Assahoun,Néant,"{Lundi,Mardi,Mercredi,Jeudi,Vendredi,Samedi,Di...",Nsp,0.908986,6.471479,maritime,ave,assahoun,pisciculture



[RETENUES_EAU]  retenues_eau.csv
  Doublons FID : 0 ✓
  Doublons géométrie : 0 ✓
  Lignes : 611 → 611 (supprimé 0)
  Géométries : {'MultiPolygon': 611}
  ✓ Toutes les géométries dans les bornes du Togo
  Colonnes finales : ['FID', 'region_nom_bdd', 'prefecture_nom_bdd', 'commune_nom_bdd', 'canton_nom_bdd', 'village', 'barrage_nom_officiel', 'barrage_type', 'barrage_utilisation', 'geometry', 'centroid_lon', 'centroid_lat', 'region_nom_bdd_norm', 'prefecture_nom_bdd_norm', 'canton_nom_bdd_norm', 'source']


,FID,region_nom_bdd,prefecture_nom_bdd,commune_nom_bdd,canton_nom_bdd,village,barrage_nom_officiel,barrage_type,barrage_utilisation,centroid_lon,centroid_lat,region_nom_bdd_norm,prefecture_nom_bdd_norm,canton_nom_bdd_norm,source
0,_mview_general_retenues_eau_collinaires.fid-54...,Maritime,Avé,Avé 1,Ando,Adekpui,Nsp,Retenue d'eau,{autre},0.805842,6.478082,maritime,ave,ando,retenues_eau
1,_mview_general_retenues_eau_collinaires.fid-54...,Maritime,Avé,Avé 1,Kévé,Seve-kpota,Nsp,Retenue d'eau,"{elevage,irrigation}",0.962201,6.434850,maritime,ave,keve,retenues_eau
2,_mview_general_retenues_eau_collinaires.fid-54...,Maritime,Golfe,Golfe 4,Amoutivé,Nyekonakpoé,Barrage Togbato de Nyekonakpoé,Retenue d'eau,"{irrigation,elevage}",1.206113,6.136881,maritime,golfe,amoutive,retenues_eau



[BARRAGES]  barrages.csv
  Doublons FID : 0 ✓
  Doublons géométrie : 0 ✓
  Lignes : 300 → 300 (supprimé 0)
  Géométries : {'MultiLineString': 300}
  ✓ Toutes les géométries dans les bornes du Togo
  Colonnes finales : ['FID', 'region_nom_bdd', 'prefecture_nom_bdd', 'commune_nom_bdd', 'canton_nom_bdd', 'geometry', 'centroid_lon', 'centroid_lat', 'region_nom_bdd_norm', 'prefecture_nom_bdd_norm', 'canton_nom_bdd_norm', 'source']


,FID,region_nom_bdd,prefecture_nom_bdd,commune_nom_bdd,canton_nom_bdd,centroid_lon,centroid_lat,region_nom_bdd_norm,prefecture_nom_bdd_norm,canton_nom_bdd_norm,source
0,_mview_general_digues_petits_barrages.fid-54b5...,Maritime,Avé,Avé 2,Badja,1.027890,6.362503,maritime,ave,badja,barrages
1,_mview_general_digues_petits_barrages.fid-54b5...,Maritime,Avé,Avé 2,Badja,1.007272,6.343431,maritime,ave,badja,barrages
2,_mview_general_digues_petits_barrages.fid-54b5...,Maritime,Avé,Avé 1,Edzi,0.870935,6.400912,maritime,ave,edzi,barrages


## 2. Vérification croisée des noms de préfectures

On vérifie que les noms dans les sources correspondent aux 39 préfectures officielles du Togo.

In [4]:
# Référentiel officiel des 5 régions et 39 préfectures du Togo
# Source : Wikipedia / INSEED Togo (loi N° 2019-006)
# Total : 7 + 7 + 5 + 12 + 8 = 39 préfectures
PREFECTURES_OFFICIELLES = {
    'Savanes':  ['Cinkassé', 'Kpendjal', 'Kpendjal-Ouest', 'Oti', 'Oti-Sud', 'Tandjoaré', 'Tône'],   # 7
    'Kara':     ['Assoli', 'Bassar', 'Binah', 'Dankpen', 'Doufelgou', 'Kéran', 'Kozah'],               # 7
    'Centrale': ['Blitta', 'Mô', 'Sotouboua', 'Tchamba', 'Tchaoudjo'],                                 # 5
    'Plateaux': ['Agou', 'Akébou', 'Amou', 'Anié', 'Danyi', 'Est-Mono', 'Haho',
                 'Kloto', 'Kpélé', 'Moyen-Mono', 'Ogou', 'Wawa'],                                      # 12
    'Maritime': ['Agoè-Nyivé', 'Avé', 'Bas-Mono', 'Golfe', 'Lacs', 'Vo', 'Yoto', 'Zio'],             # 8
}

# Préfectures trouvées dans les données
prefs_trouvees = set()
for name, gdf in geo_clean.items():
    if 'prefecture_nom_bdd' in gdf.columns:
        prefs_trouvees.update(gdf['prefecture_nom_bdd'].dropna().unique())

print(f'Préfectures trouvées dans les données ({len(prefs_trouvees)}) :')
for p in sorted(prefs_trouvees):
    print(f'  {p}')

# Vérifier les noms normalisés
prefs_norm_data     = {normalize_name(p) for p in prefs_trouvees}
prefs_norm_officiel = {normalize_name(p) for preflist in PREFECTURES_OFFICIELLES.values() for p in preflist}

non_trouvees = prefs_norm_officiel - prefs_norm_data
if non_trouvees:
    print(f'\n⚠ Préfectures officielles absentes des données : {non_trouvees}')
else:
    print(f'\n✓ Toutes les préfectures sont présentes dans les données')

# Noms dans les données qui ne matchent pas l'officiel
inconnues = prefs_norm_data - prefs_norm_officiel
if inconnues:
    print(f'\n⚠ Noms non reconnus dans les données : {inconnues}')
    print('  → À corriger manuellement dans le mapping ci-dessous')

Préfectures trouvées dans les données (39) :
  Agou
  Agoè-Nyivé
  Akébou
  Amou
  Anié
  Assoli
  Avé
  Bas-Mono
  Bassar
  Binah
  Blitta
  Cinkassé
  Dankpen
  Danyi
  Doufelgou
  Est-Mono
  Golfe
  Haho
  Kloto
  Kozah
  Kpendjal
  Kpendjal-Ouest
  Kpélé
  Kéran
  Lacs
  Moyen-Mono
  Mô
  Ogou
  Oti
  Oti-Sud
  Sotouboua
  Tandjoaré
  Tchamba
  Tchaoudjo
  Tône
  Vo
  Wawa
  Yoto
  Zio

✓ Toutes les préfectures sont présentes dans les données


In [5]:
# Mapping de correction des noms si nécessaire
# Ajouter ici les corrections identifiées à l'étape précédente
# Format : 'nom_dans_données' : 'nom_officiel'
CORRECTIONS_PREFS = {
    # Exemples à adapter selon les résultats :
    # 'agoe-nyive' : 'Agoè-Nyivé',
    # 'kpendjal ouest' : 'Kpendjal-Ouest',
}

def corriger_prefecture(nom_norm):
    return CORRECTIONS_PREFS.get(nom_norm, nom_norm)

# Appliquer les corrections
for name, gdf in geo_clean.items():
    if 'prefecture_nom_bdd_norm' in gdf.columns:
        geo_clean[name]['prefecture_nom_bdd_clean'] = (
            gdf['prefecture_nom_bdd_norm'].apply(corriger_prefecture)
        )

print('Corrections appliquées ✓')

Corrections appliquées ✓


## 3. Séries Banque mondiale — nettoyage

In [6]:
BM_FILES = {
    'agri_rural':     RAW / 'agri_rural.csv',
    'va_pib':         RAW / 'va_pib.csv',
    'va_croissance':  RAW / 'va_croissance.csv',
    'va_usd':         RAW / 'va_usd.csv',
    'va_travailleur': RAW / 'va_travailleur.csv',
}

bm_clean = {}

for name, path in BM_FILES.items():
    print(f'\n{"─"*50}')
    print(f'[{name.upper()}]')

    df, enc, sep = read_csv_smart(path)

    # ── Supprimer ligne de métadonnées HDX (#country+name...) ────
    if df.iloc[0].astype(str).str.startswith('#').any():
        df = df.iloc[1:].reset_index(drop=True)
        print(f'  Ligne métadonnées supprimée')

    # ── Standardiser les noms de colonnes ─────────────────────────
    df.columns = df.columns.str.lower().str.strip()

    # ── Renommer selon le format détecté ─────────────────────────
    if 'date' in df.columns:
        df = df.rename(columns={'date': 'year'})
    if 'indicator' in df.columns and name != 'agri_rural':
        df = df.rename(columns={'indicator': 'indicator_name'})

    # ── Convertir types ───────────────────────────────────────────
    if 'year' in df.columns:
        df['year'] = pd.to_numeric(df['year'], errors='coerce').astype('Int64')
    if 'value' in df.columns:
        df['value'] = pd.to_numeric(df['value'], errors='coerce')

    # ── Supprimer lignes sans valeur ──────────────────────────────
    n_avant = len(df)
    df = df.dropna(subset=['value'])
    n_apres = len(df)
    if n_avant != n_apres:
        print(f'  Lignes sans valeur supprimées : {n_avant - n_apres}')

    # ── Statistiques ──────────────────────────────────────────────
    if 'year' in df.columns:
        annees = sorted(df['year'].dropna().unique())
        print(f'  Années : {int(annees[0])} → {int(annees[-1])} ({len(annees)} valeurs)')
    if 'value' in df.columns:
        print(f'  Valeur min/max : {df["value"].min():.2f} / {df["value"].max():.2f}')

    # ── Indicateurs disponibles (agri_rural contient plusieurs) ───
    if 'indicator name' in df.columns:
        nb_ind = df['indicator name'].nunique()
        print(f'  Indicateurs distincts : {nb_ind}')
        print(df['indicator name'].unique()[:5])
    elif 'indicator_name' in df.columns:
        print(f'  Indicateur : {df["indicator_name"].iloc[0][:80]}...')

    bm_clean[name] = df
    display(df.head(3))


──────────────────────────────────────────────────
[AGRI_RURAL]
  Ligne métadonnées supprimée
  Années : 1960 → 2023 (64 valeurs)
  Valeur min/max : -1.28 / 1662826080.14
  Indicateurs distincts : 36
<StringArray>
[          'Fertilizer consumption (% of fertilizer production)',
 'Fertilizer consumption (kilograms per hectare of arable land)',
                                    'Agricultural land (sq. km)',
                            'Agricultural land (% of land area)',
                                        'Arable land (hectares)']
Length: 5, dtype: str


,country name,country iso3,year,indicator name,indicator code,value
0,Togo,TGO,2016,Fertilizer consumption (% of fertilizer produc...,AG.CON.FERT.PT.ZS,14.214588
1,Togo,TGO,2015,Fertilizer consumption (% of fertilizer produc...,AG.CON.FERT.PT.ZS,2.939040
2,Togo,TGO,2014,Fertilizer consumption (% of fertilizer produc...,AG.CON.FERT.PT.ZS,1.359011



──────────────────────────────────────────────────
[VA_PIB]
  Lignes sans valeur supprimées : 3
  Années : 1963 → 2023 (61 valeurs)
  Valeur min/max : 14.32 / 56.17
  Indicateur : Agriculture, forestry, and fishing, value added (% of GDP)...


,indicator_name,country,countryiso3code,year,value,unit,obs_status,decimal
0,"Agriculture, forestry, and fishing, value adde...",Togo,TGO,2023,18.130832,NaN,NaN,1
1,"Agriculture, forestry, and fishing, value adde...",Togo,TGO,2022,18.700143,NaN,NaN,1
2,"Agriculture, forestry, and fishing, value adde...",Togo,TGO,2021,18.364991,NaN,NaN,1



──────────────────────────────────────────────────
[VA_CROISSANCE]
  Lignes sans valeur supprimées : 6
  Années : 1966 → 2023 (58 valeurs)
  Valeur min/max : -8.84 / 29.91
  Indicateur : Agriculture, forestry, and fishing, value added (annual % growth)...


,indicator_name,country,countryiso3code,year,value,unit,obs_status,decimal
0,"Agriculture, forestry, and fishing, value adde...",Togo,TGO,2023,4.154855,NaN,NaN,1
1,"Agriculture, forestry, and fishing, value adde...",Togo,TGO,2022,5.103109,NaN,NaN,1
2,"Agriculture, forestry, and fishing, value adde...",Togo,TGO,2021,3.288445,NaN,NaN,1



──────────────────────────────────────────────────
[VA_USD]
  Lignes sans valeur supprimées : 5
  Années : 1965 → 2023 (59 valeurs)
  Valeur min/max : 225341484.17 / 1544291120.27
  Indicateur : Agriculture, forestry, and fishing, value added (constant 2015 US$)...


,indicator_name,country,countryiso3code,year,value,unit,obs_status,decimal
0,"Agriculture, forestry, and fishing, value adde...",Togo,TGO,2023,1.544291e+09,NaN,NaN,0
1,"Agriculture, forestry, and fishing, value adde...",Togo,TGO,2022,1.482688e+09,NaN,NaN,0
2,"Agriculture, forestry, and fishing, value adde...",Togo,TGO,2021,1.410698e+09,NaN,NaN,0



──────────────────────────────────────────────────
[VA_TRAVAILLEUR]
  Lignes sans valeur supprimées : 32
  Années : 1991 → 2022 (32 valeurs)
  Valeur min/max : 673.62 / 1592.16
  Indicateur : Agriculture, forestry, and fishing, value added per worker (constant 2015 US$)...


,indicator_name,country,countryiso3code,year,value,unit,obs_status,decimal
1,"Agriculture, forestry, and fishing, value adde...",Togo,TGO,2022,1592.158243,NaN,NaN,2
2,"Agriculture, forestry, and fishing, value adde...",Togo,TGO,2021,1543.266447,NaN,NaN,2
3,"Agriculture, forestry, and fishing, value adde...",Togo,TGO,2020,1522.587688,NaN,NaN,2


## 4. Export vers `data/processed/`

In [7]:
print('Export des fichiers nettoyés...\n')

# ── Sources géospatiales → CSV avec centroïdes ────────────────────────────
for name, gdf in geo_clean.items():
    out = PROC / f'{name}_clean.csv'
    # Sauvegarder sans la géométrie WKT (trop lourde) mais avec centroïdes
    df_export = gdf.drop(columns='geometry').copy()
    df_export.to_csv(out, index=False, encoding='utf-8')
    print(f'  ✓ {out.name}  ({len(df_export)} lignes)')

# ── Sauvegarder aussi en GeoJSON pour les cartes ─────────────────────────
for name, gdf in geo_clean.items():
    out = PROC / f'{name}_clean.geojson'
    gdf.to_file(out, driver='GeoJSON')
    print(f'  ✓ {out.name}  ({len(gdf)} entités)')

# ── Séries Banque mondiale ────────────────────────────────────────────────
for name, df in bm_clean.items():
    out = PROC / f'{name}_clean.csv'
    df.to_csv(out, index=False, encoding='utf-8')
    print(f'  ✓ {out.name}  ({len(df)} lignes)')

print('\nTous les fichiers exportés dans data/processed/ ✓')

Export des fichiers nettoyés...

  ✓ elevage_clean.csv  (3004 lignes)
  ✓ abattoirs_clean.csv  (36 lignes)
  ✓ pisciculture_clean.csv  (141 lignes)
  ✓ retenues_eau_clean.csv  (611 lignes)
  ✓ barrages_clean.csv  (300 lignes)
  ✓ elevage_clean.geojson  (3004 entités)
  ✓ abattoirs_clean.geojson  (36 entités)
  ✓ pisciculture_clean.geojson  (141 entités)
  ✓ retenues_eau_clean.geojson  (611 entités)
  ✓ barrages_clean.geojson  (300 entités)
  ✓ agri_rural_clean.csv  (1666 lignes)
  ✓ va_pib_clean.csv  (61 lignes)
  ✓ va_croissance_clean.csv  (58 lignes)
  ✓ va_usd_clean.csv  (59 lignes)
  ✓ va_travailleur_clean.csv  (32 lignes)

Tous les fichiers exportés dans data/processed/ ✓


## 5. Résumé Phase 2

In [8]:
print('=' * 55)
print('RÉSUMÉ PHASE 2 — NETTOYAGE')
print('=' * 55)

print('\n[SOURCES GÉOSPATIALES]')
for name, gdf in geo_clean.items():
    geom = gdf.geom_type.mode()[0]
    print(f'  {name:15s} | {len(gdf):4d} entités | {geom:15s} | centroïdes ✓')

print('\n[SÉRIES BANQUE MONDIALE]')
for name, df in bm_clean.items():
    print(f'  {name:15s} | {len(df):4d} lignes | value nulls: {df["value"].isna().sum()}')

print('\n[CLÉS DE JOINTURE ADMIN DISPONIBLES]')
print('  → region_nom_bdd      (5 régions)')
print('  → prefecture_nom_bdd  (39 préfectures)')
print('  → canton_nom_bdd      (~394 cantons)')
print()
print('Phase 2 terminée → Prochaine étape : 03_eda.ipynb')

RÉSUMÉ PHASE 2 — NETTOYAGE

[SOURCES GÉOSPATIALES]
  elevage         | 3004 entités | Polygon         | centroïdes ✓
  abattoirs       |   36 entités | Point           | centroïdes ✓
  pisciculture    |  141 entités | MultiPolygon    | centroïdes ✓
  retenues_eau    |  611 entités | MultiPolygon    | centroïdes ✓
  barrages        |  300 entités | MultiLineString | centroïdes ✓

[SÉRIES BANQUE MONDIALE]
  agri_rural      | 1666 lignes | value nulls: 0
  va_pib          |   61 lignes | value nulls: 0
  va_croissance   |   58 lignes | value nulls: 0
  va_usd          |   59 lignes | value nulls: 0
  va_travailleur  |   32 lignes | value nulls: 0

[CLÉS DE JOINTURE ADMIN DISPONIBLES]
  → region_nom_bdd      (5 régions)
  → prefecture_nom_bdd  (39 préfectures)
  → canton_nom_bdd      (~394 cantons)

Phase 2 terminée → Prochaine étape : 03_eda.ipynb
